# CareTrace — Neurosymbolic Pediatric Triage

**UC Berkeley DATASCI 290 — Neurosymbolic AI (Spring 2026 Final Project)**

**Team:** Annas (Orchestrate) · Yoko (Extract) · David (Knowledge Graph) · Alex (Rules)

---

## What this notebook shows

CareTrace is a pediatric fever-triage agent that combines four stacked layers:

1. **LLM interpretation** — pulls structured clinical facts out of a caregiver's natural-language message
2. **Knowledge graph grounding** — maps surface symptoms to SNOMED CT concepts and red-flag rules (Neo4j, with a dict fallback)
3. **Symbolic rules** — a 3-layer plain-Python pipeline (observation → concern → decision) encoding Seattle Children's Fever CPG
4. **LLM explanation** — turns the structured decision into empathetic caregiver-facing guidance

The LLM never makes the clinical decision. It extracts facts on the way in and verbalizes the rules' decision on the way out. Everything safety-critical happens in symbolic code, so every disposition is fully traceable.

## Architecture

```
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌─────────────┐
│ Caregiver   │───▶│ Interpret    │───▶│ Normalize     │───▶│ Evaluate    │
│ message     │    │ (LLM → facts)│    │ (KG + flags)  │    │ (rules)     │
└─────────────┘    └──────────────┘    └───────────────┘    └──────┬──────┘
                                                                   │
                                        ┌──────────────────────────┴─┐
                                        │ disposition?                │
                                        ├─────────────────┬───────────┤
                                        ▼                 ▼
                                ┌───────────────┐  ┌────────────────┐
                                │ Explain       │  │ Ask follow-up  │
                                │ (LLM verbalize)│  │ (structured Q) │
                                └───────┬───────┘  └────────┬───────┘
                                        │                   │
                                        └────────┬──────────┘
                                                 ▼
                                         Caregiver reply

Dispositions (Alex's vocab):  er_now  |  urgent_eval  |  home_monitor  |  unsupported
```


## 0. Setup

Import the consolidated `caretrace` package and load the Groq API key. The KG adapter auto-detects Neo4j; if it's not reachable we fall back to an in-memory dict of SNOMED concepts.

In [1]:
import os, sys, json
from pprint import pprint
from pathlib import Path

# Make sure we can import the caretrace package from this notebook's parent
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Load .env so Groq + Neo4j credentials become available
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

print("GROQ_API_KEY set:     ", bool(os.getenv("GROQ_API_KEY")))
print("NEO4J_URI set:        ", bool(os.getenv("NEO4J_URI")))
print("KG backend preference:", os.getenv("CARETRACE_KG_BACKEND", "auto"))

GROQ_API_KEY set:      True
NEO4J_URI set:         True
KG backend preference: auto


## 1. Shared state — `ClinicalState`

Every node in the LangGraph reads and writes a single `ClinicalState` TypedDict. The most important field is `facts` — a nested dict using Alex's categorical vocabulary that the rules engine consumes directly. Raw typed fields (`age_months`, `temperature_f`, `current_medication`) live at the top level because the KG and fallback red flags need them.


In [2]:
from caretrace.state import (
    ClinicalState,
    FACT_KEYS,
    REQUIRED_FACTS_FOR_HOME,
    FACT_QUESTIONS,
    DISPOSITION_SEVERITY,
    initial_state,
)

print("FACT_KEYS:              ", FACT_KEYS)
print("REQUIRED_FACTS_FOR_HOME:", REQUIRED_FACTS_FOR_HOME)
print()
print("Initial (blank) state keys:")
for k in initial_state().keys():
    print(f"  - {k}")

FACT_KEYS:               ['fever', 'alert', 'intake', 'urination', 'vomiting', 'breathing', 'seizure', 'rash']
REQUIRED_FACTS_FOR_HOME: ['alert', 'breathing', 'intake', 'urination']

Initial (blank) state keys:
  - messages
  - turn
  - facts
  - age_months
  - temperature_f
  - fever_duration_days
  - current_medication
  - medication_last_dose
  - weight_kg
  - raw_symptoms
  - kg_backend
  - grounded_concepts
  - all_sctids
  - kg_red_flags
  - observation_predicates
  - concern_predicates
  - decision
  - rules_triggered
  - disposition
  - missing_required
  - follow_up_question
  - explanation
  - key_positives
  - key_negatives
  - is_complete
  - phase


In [3]:
# Disposition severity ranking — 'merge worst wins' semantics
print("Disposition priority (lower = more severe):")
for disp, rank in sorted(DISPOSITION_SEVERITY.items(), key=lambda x: x[1]):
    print(f"  {rank}: {disp}")

Disposition priority (lower = more severe):
  0: er_now
  1: urgent_eval
  2: home_monitor
  3: unsupported
  4: None


## 2. Knowledge graph adapter

`KnowledgeAdapter` wraps David's `KnowledgeRetrievalAgent` and falls back to a dict of SNOMED concepts when Neo4j isn't reachable. Both backends expose the same `ground_symptoms(mentions) → {results, all_sctids}` and `get_red_flags(sctids) → [...]` shape, so the rest of the pipeline is backend-agnostic.


In [4]:
from caretrace.kg.adapter import KnowledgeAdapter

# prefer='auto' — tries Neo4j, falls back to dict
adapter = KnowledgeAdapter(prefer="auto")
print(f"Active backend: {adapter.backend}")

Active backend: neo4j


In [5]:
# Ground a set of mentions. The adapter walks IS_A ancestors and returns
# SCTIDs for every matched concept.
mentions = ["fever", "lethargy", "vomiting", "reduced fluid intake"]
grounding = adapter.ground_symptoms(mentions)

print("Grounded concepts:")
for r in grounding["results"]:
    print(f"  {r.get('mention')!r:30s} → {r.get('concepts', [])[:2]}")
print()
print(f"All SCTIDs: {grounding['all_sctids'][:10]}{'...' if len(grounding['all_sctids']) > 10 else ''}")
print(f"Ungrounded: {grounding.get('ungrounded', [])}")

Grounded concepts:
  'fever'                        → [{'sctid': '386661006', 'fsn': 'Fever (finding)', 'type': 'finding', 'depth': 0, 'ancestors': [{'fsn': None, 'depth': None, 'sctid': None, 'type': None}]}, {'sctid': '248425001', 'fsn': 'Febrile convulsion (disorder)', 'type': 'disorder', 'depth': 0, 'ancestors': [{'fsn': 'Fever (finding)', 'depth': 1, 'sctid': '386661006', 'type': 'finding'}]}]
  'lethargy'                     → []
  'vomiting'                     → [{'sctid': '422400008', 'fsn': 'Vomiting (disorder)', 'type': 'disorder', 'depth': 0, 'ancestors': [{'fsn': None, 'depth': None, 'sctid': None, 'type': None}]}]
  'reduced fluid intake'         → []

All SCTIDs: ['422400008', '248425001', '386661006']
Ungrounded: ['lethargy', 'reduced fluid intake']


In [6]:
# Red flags from the KG, keyed on the grounded SCTIDs
kg_flags = adapter.get_red_flags(grounding["all_sctids"])
print(f"KG red flags for these concepts ({len(kg_flags)}):")
for f in kg_flags:
    print(f"  [{f.get('source', 'kg'):8s}] {f['rule_id']:20s} → {f['disposition']:12s}  {f.get('description', '')[:60]}")

KG red flags for these concepts (3):
  [neo4j   ] RF_002               → er_now        Febrile seizure
  [neo4j   ] RF_001               → er_now        Fever in infant under 3 months
  [neo4j   ] RF_009               → urgent_eval   Vomiting preventing oral rehydration


## 3. Fallback red flags

When Neo4j is offline, Alex's rules alone miss high-severity signals like infant fever, seizure, and breathing difficulty (because those aren't in the dehydration-focused fever CPG). `check_fallback_red_flags` runs a small dict-based rule set that catches them and emits the same red-flag shape the safety layer expects.


In [7]:
from caretrace.kg.fallback_red_flags import check_fallback_red_flags, highest_disposition

# Build a few sample states showing each fallback rule firing
samples = [
    ("2mo with any fever",
        {"age_months": 2, "facts": {"fever": "yes"}}),
    ("5yo with seizure",
        {"age_months": 60, "facts": {"fever": "yes", "seizure": "yes"}}),
    ("4yo with breathing difficulty",
        {"age_months": 48, "facts": {"fever": "yes", "breathing": "difficulty"}}),
    ("Temperature >= 104°F",
        {"age_months": 60, "temperature_f": 104.2, "facts": {"fever": "yes"}}),
    ("Rash + fever",
        {"age_months": 60, "facts": {"fever": "yes", "rash": "yes"}}),
]

for label, state in samples:
    flags = check_fallback_red_flags(state)
    print(f"— {label}")
    for f in flags:
        print(f"    {f['rule_id']:20s} → {f['disposition']:12s}  ({f['description']})")
    print(f"    highest: {highest_disposition(flags)}")
    print()

— 2mo with any fever
    FB_INFANT_FEVER      → er_now        (Any fever in an infant under 3 months is an emergency.)
    highest: er_now

— 5yo with seizure
    FB_SEIZURE           → er_now        (A febrile seizure requires immediate emergency evaluation.)
    highest: er_now

— 4yo with breathing difficulty
    FB_BREATHING         → er_now        (Difficulty breathing in a febrile child is an emergency.)
    highest: er_now

— Temperature >= 104°F
    FB_VERY_HIGH_FEVER   → urgent_eval   (Temperature ≥104°F warrants urgent evaluation.)
    highest: urgent_eval

— Rash + fever
    FB_RASH_FEVER        → urgent_eval   (New rash with fever — check for non-blanching spots urgently.)
    highest: urgent_eval



## 4. Interpretation agent — LLM → structured facts

The interpretation agent uses Groq's Llama 3.3 70B with Pydantic `with_structured_output` to extract clinical facts from a single caregiver message. It outputs Yoko's binary vocabulary (`alert: yes/no`, `drinking: yes/some/no`), then `_translate_to_facts` maps those onto Alex's categorical vocabulary. The LLM is forbidden from giving any advice, diagnosis, or disposition — its only job is extraction.


In [8]:
from langchain_core.messages import HumanMessage
from caretrace.agents.interpretation import interpret, ExtractionResult, _translate_to_facts, _derive_raw_symptoms

# Build a fresh state with one caregiver message
state = initial_state()
state["messages"] = [HumanMessage(content=(
    "My 6-year-old has had a fever since yesterday. Temperature is 101.8. "
    "He's awake and talking to me, drinking water normally, breathing fine. "
    "He threw up once at dinner."
))]

updates = interpret(state)
print("Raw fields extracted:")
for k in ("age_months", "temperature_f", "fever_duration_days", "current_medication"):
    if k in updates:
        print(f"  {k}: {updates[k]}")
print()
print("Translated facts:")
for k, v in updates["facts"].items():
    print(f"  {k}: {v}")
print()
print(f"Raw symptoms (for KG grounding): {updates['raw_symptoms']}")
print(f"Missing required facts: {updates['missing_required']}")
print(f"Follow-up question: {updates.get('follow_up_question')}")

Raw fields extracted:
  age_months: 72
  temperature_f: 101.8
  fever_duration_days: 1.0

Translated facts:
  fever: yes
  alert: normal
  intake: normal
  vomiting: once
  breathing: normal

Raw symptoms (for KG grounding): ['fever', 'vomiting']
Missing required facts: ['urination']
Follow-up question: Has your child urinated in the last 8 hours?


## 5. Symbolic rules engine

Alex's 3-layer pipeline lives in `src/rules/rules_agent.py`. It's plain Python — no pyDatalog — for ease of debugging and introspection.

- **Layer 1 (observation):** maps raw facts to predicates like `fever_present`, `poor_intake`, `no_urine`, `lethargy`
- **Layer 2 (concern):** combines observations. Key concern is `dehydration_concern`, which scores `no_urine`/`no_intake` at +2 and `poor_intake`/`repeated_vomiting` at +1. Fever amplifies the score.
- **Layer 3 (decision):** routes concerns to `er_now` | `urgent_eval` | `home_monitor` | `unsupported`

Any `danger_red_flag` (lethargy, breathing difficulty, seizure) short-circuits directly to `er_now`.


In [9]:
from src.rules import rules_agent

# Case A: mild fever, normal everything → home_monitor
case_a = {"facts": {"fever": "yes", "alert": "normal", "breathing": "normal", "intake": "normal", "urination": "normal"}}
out = rules_agent(case_a)
print("Case A — mild fever, no concerns:")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")
print()

# Case B: reduced intake + fever → dehydration amplification → urgent_eval
case_b = {"facts": {"fever": "yes", "alert": "normal", "breathing": "normal", "intake": "reduced", "urination": "normal"}}
out = rules_agent(case_b)
print("Case B — reduced intake + fever (dehydration amplification):")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")
print()

# Case C: lethargy → danger_red_flag → er_now
case_c = {"facts": {"fever": "yes", "alert": "reduced", "breathing": "normal", "intake": "none", "urination": "none"}}
out = rules_agent(case_c)
print("Case C — lethargy + no intake + no urine:")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")

Case A — mild fever, no concerns:
  observations: ['fever_present']
  concerns:     []
  decision:     home_monitor

Case B — reduced intake + fever (dehydration amplification):
  observations: ['fever_present', 'poor_intake']
  concerns:     ['dehydration_concern']
  decision:     urgent_eval

Case C — lethargy + no intake + no urine:
  observations: ['fever_present', 'lethargy', 'no_intake', 'no_urine']
  concerns:     ['danger_red_flag', 'dehydration_concern']
  decision:     er_now


## 6. Safety merge — rules + KG red flags + validation

`evaluate_rules` runs Alex's rules on the facts, then merges with any KG or fallback red flags. Because David's Neo4j synonym matcher uses substring containment, grounding `"fever"` can pull in adjacent concepts like `"febrile seizure"`, spuriously firing red flags like RF_001 (infant fever) or RF_002 (febrile seizure).

To prevent these false positives while still honoring legitimate KG signals, each KG-sourced flag must pass a per-`rule_id` predicate against the actual facts. **Fallback-sourced flags are trusted as-is** since they already checked the facts before being emitted. **Unknown rule_ids fail safe** (rejected) so that schema drift can't silently escalate.


In [10]:
from caretrace.agents.safety import evaluate_rules, _RED_FLAG_VALIDATORS, _validate_red_flag

# Simulate a state where KG spuriously returned RF_001 (infant fever) for a 6yo
state = initial_state()
state["age_months"] = 72  # 6 years old
state["facts"] = {"fever": "yes", "alert": "normal", "breathing": "normal",
                  "intake": "normal", "urination": "normal"}

# Spurious KG red flags (as would happen with substring-matching fever → febrile seizure)
state["kg_red_flags"] = [
    {"rule_id": "RF_001", "description": "Infant fever", "disposition": "er_now", "source": "neo4j"},
    {"rule_id": "RF_002", "description": "Febrile seizure", "disposition": "er_now", "source": "neo4j"},
]

print("Raw KG red flags (before validation):")
for f in state["kg_red_flags"]:
    print(f"  {f['rule_id']} → {f['disposition']}")

print("\nValidation results:")
for f in state["kg_red_flags"]:
    ok = _validate_red_flag(f, state)
    print(f"  {f['rule_id']}: {'APPLY' if ok else 'FILTERED (predicate failed)'}")

print("\nFinal disposition after evaluate_rules:")
out = evaluate_rules(state)
print(f"  disposition: {out['disposition']}")
print(f"  rules_triggered: {out['rules_triggered']}")

Raw KG red flags (before validation):
  RF_001 → er_now
  RF_002 → er_now

Validation results:
  RF_001: FILTERED (predicate failed)
  RF_002: FILTERED (predicate failed)

Final disposition after evaluate_rules:
  disposition: home_monitor
  rules_triggered: ['obs:fever_present', 'safe:no_red_flags', 'decision:home_monitor']


## 7. Explanation agent — structured decision → caregiver text

The explanation agent never makes clinical decisions. It receives a fully-formed decision object from the safety layer (disposition + rule trace + positives + negatives) and asks the LLM to verbalize it in empathetic, plain language the caregiver can act on.


In [11]:
from caretrace.agents.explanation import explain

# Use the Case C result from Section 5 as input to explanation
state = initial_state()
state["age_months"] = 72
state["temperature_f"] = 103.5
state["facts"] = {"fever": "yes", "alert": "reduced", "intake": "none",
                  "breathing": "normal", "urination": "none"}
state.update(evaluate_rules(state))

# Now ask the explanation agent to verbalize
result = explain(state)

# explain() returns an updates dict including a messages list with the AIMessage
for m in result.get("messages", []):
    if hasattr(m, "content"):
        print(m.content)

I'm so sorry to hear that your child is not feeling well. I strongly recommend that you take them to the Emergency Room right now. The combination of a high fever, lethargy, refusal to drink fluids, and not having urinated in the last 8 hours are all very concerning signs that require immediate medical attention. These symptoms suggest that your child may be at risk for severe dehydration, which can be life-threatening if not treated promptly.

Please watch for any signs of worsening condition, such as difficulty breathing, severe headache, or seizures, and get to the ER as quickly as possible.

I know this is scary, but getting your child to the hospital right away is the best way to ensure they receive the care they need. The doctors and nurses at the ER will be able to assess and treat your child's condition, and you'll be able to get the support and guidance you need during this challenging time.


## 8. Scenario 1 — Moderate fever, alert, drinking → `home_monitor`

Full multi-turn conversation through the compiled LangGraph, including the LLM interpretation, KG grounding, rule evaluation, and LLM explanation. Watch how the system asks follow-up questions until it has the required facts, then issues a `home_monitor` disposition.


In [12]:
from caretrace.graph import create_app
from caretrace.state import initial_state
from langchain_core.messages import HumanMessage, AIMessage

def render_turn(user_msg, result):
    print(f"\n\x1b[1m[USER]\x1b[0m  {user_msg}")
    for m in reversed(result.get("messages", [])):
        if isinstance(m, AIMessage):
            print(f"\n\x1b[1m[CARETRACE]\x1b[0m")
            for line in m.content.splitlines():
                print(f"  {line}")
            break
    print(f"\n  disposition: \x1b[93m{result.get('disposition') or 'undecided'}\x1b[0m")
    print(f"  phase:       {result.get('phase')}")
    print(f"  facts:       {result.get('facts')}")
    missing = result.get('missing_required', [])
    if missing:
        print(f"  missing:     {missing}")
    rules = result.get('rules_triggered', [])
    if rules:
        print(f"  rule trace:  {rules}")
    flags = result.get('kg_red_flags', [])
    if flags:
        print(f"  red flags:")
        for f in flags:
            print(f"    [{f.get('source', 'kg')}] {f['rule_id']} → {f['disposition']}")

# Run Scenario 1
app, config = create_app(thread_id="nb-s1")
state = initial_state()

scenario_1 = [
    "Hi, my 6-year-old has had a fever since yesterday and threw up once at dinner. I'm worried.",
    "Temperature is 101.8 right now. He's awake and talking to me normally, breathing fine. "
    "He's drinking water like he usually does, just not eating as much solid food.",
    "He just peed about 3 hours ago, normal amount. He's on amoxicillin for an ear infection.",
]

for msg in scenario_1:
    state["messages"] = [HumanMessage(content=msg)]
    result = app.invoke(state, config)
    state.update(result)
    render_turn(msg, result)
    if result.get("is_complete"):
        break

print(f"\n\x1b[1m\x1b[92mFINAL: {state.get('disposition')}\x1b[0m")


[USER]  Hi, my 6-year-old has had a fever since yesterday and threw up once at dinner. I'm worried.

[CARETRACE]
  "I'm so glad you're reaching out for help with your little one. I just want to make sure they're doing okay right now. Can you tell me, is your child awake and able to respond to you when you talk to them, and are they breathing normally - not having any trouble catching their breath or breathing really quickly?"

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once'}
  missing:     ['alert', 'breathing', 'intake', 'urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']
  red flags:
    [neo4j] RF_002 → er_now
    [neo4j] RF_001 → er_now
    [neo4j] RF_009 → urgent_eval



[USER]  Temperature is 101.8 right now. He's awake and talking to me normally, breathing fine. He's drinking water like he usually does, just not eating as much solid food.

[CARETRACE]
  "I'm so glad to hear that your child is still alert and drinking normally, that's really reassuring. I just have a few more questions to help me understand what's going on. Can you tell me, has your child had a wet diaper or used the bathroom to urinate in the last 8 hours? This will help me make sure they're staying hydrated and that their body is functioning properly."

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once', 'alert': 'normal', 'intake': 'normal', 'breathing': 'normal'}
  missing:     ['urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']
  red flags:
    [neo4j] RF_002 → er_now
    [neo4j] RF_001 → er_now
    [neo4j] RF_009 → urgent_eval



[USER]  He just peed about 3 hours ago, normal amount. He's on amoxicillin for an ear infection.

[CARETRACE]
  You can safely monitor your child at home. The reason for this is that despite having a fever, your child is awake, responding normally, breathing normally, drinking fluids, and urinating as expected. These are all good signs that your child is not in immediate danger.
  
  However, it's crucial to watch for any signs that might require immediate ER attention, such as a febrile seizure (which is a seizure caused by fever), severe vomiting that prevents your child from keeping fluids down, or if your child shows signs of severe distress or difficulty breathing.
  
  For tonight, make sure your child stays hydrated by offering plenty of fluids, like water or clear broth. You can also help manage the fever with medication like acetaminophen (following the correct dosage for your child's age and weight). Encourage rest and keep a close eye on your child's condition.
  
  Remembe

## 9. Scenario 2 — High fever, reduced alertness, refusing fluids → `er_now`

Same pipeline, different inputs. The `lethargy` observation triggers `danger_red_flag` directly in the rules layer, yielding `er_now` as soon as the caregiver reports reduced responsiveness.


In [13]:
app, config = create_app(thread_id="nb-s2")
state = initial_state()

scenario_2 = [
    "My 6-year-old is really sick. High fever and she threw up once.",
    "Her temperature is 103.5. She's really out of it — barely responding when I call her. "
    "She won't drink anything. Her breathing seems okay though. "
    "There's a stomach bug going around her school.",
    "I don't think she's peed in about 8 hours.",
]

for msg in scenario_2:
    state["messages"] = [HumanMessage(content=msg)]
    result = app.invoke(state, config)
    state.update(result)
    render_turn(msg, result)
    if result.get("is_complete"):
        break

print(f"\n\x1b[1m\x1b[91mFINAL: {state.get('disposition')}\x1b[0m")


[USER]  My 6-year-old is really sick. High fever and she threw up once.

[CARETRACE]
  "I'm so glad you're reaching out for help with your little one. I just want to make sure they're doing okay. Can you tell me a bit more about how they're acting right now? Are they awake and responding normally when you talk to them, and are they having any trouble breathing or seeming to breathe really fast?"

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once'}
  missing:     ['alert', 'breathing', 'intake', 'urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']
  red flags:
    [neo4j] RF_002 → er_now
    [neo4j] RF_001 → er_now
    [neo4j] RF_009 → urgent_eval



[USER]  Her temperature is 103.5. She's really out of it — barely responding when I call her. She won't drink anything. Her breathing seems okay though. There's a stomach bug going around her school.

[CARETRACE]
  I'm so sorry to hear that your child is not feeling well. I strongly recommend that you take them to the Emergency Room right now. The combination of a high fever, lethargy, and refusal to drink fluids is very concerning and could be a sign of severe dehydration or another serious condition.
  
  Even though their breathing is normal, the fact that they're not responding as they normally would and are refusing fluids is a red flag that requires immediate attention. I'm worried that if we don't get them the help they need right away, their condition could worsen.
  
  If you notice any of the following, please rush to the ER even faster: difficulty breathing, seizures, or if they become unresponsive.
  
  Please get your child to the ER as quickly and safely as possible. The

## 10. Interpretation agent evaluation — golden dataset sweep

Yoko's golden dataset (`golden_dataset.json`) has 10 `targeted_cases` that stress specific failure modes of the interpreter: implied alertness, temperature unit conversion, never-overwrite semantics, etc. Each case has a `prior_state`, a `caregiver_message`, and an `expected_extracted_fields` delta.

We run the interpretation agent on each case and compute per-field accuracy on the fields the case is asserting.


In [14]:
import json
from caretrace.agents.interpretation import interpret, ExtractionResult
from langchain_core.messages import HumanMessage

with open(REPO_ROOT / "golden_dataset.json") as fh:
    gold = json.load(fh)

# Map Yoko's raw extracted fields (what ExtractionResult produces) onto our
# new interpret() which merges translated facts. For the eval we want the raw
# ExtractionResult, so we call the LLM directly with the same prompt.
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from caretrace.agents.interpretation import EXTRACTION_SYSTEM_PROMPT

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=os.getenv("GROQ_API_KEY"))
extractor = llm.with_structured_output(ExtractionResult)

def extract_from_case(case):
    prior = case["prior_state"]
    # Build a fake history line so the model knows the case context
    history_line = (
        f"Prior known state: " +
        ", ".join(f"{k}={v}" for k, v in prior.items() if v is not None)
    )
    prompt = [
        SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
        SystemMessage(content=history_line),
        HumanMessage(content=f"Latest caregiver message: {case['caregiver_message']}"),
    ]
    return extractor.invoke(prompt)

# Fields we evaluate (drop never-used ones from the gold schema)
EVAL_FIELDS = ["alert", "temperature_f", "breathing_issues", "urination_8h",
               "drinking", "current_medication", "fever", "vomiting", "rash"]

results = []
for case in gold["targeted_cases"]:
    got = extract_from_case(case)
    exp = case["expected_extracted_fields"]
    row = {"id": case["id"], "category": case["category"]}
    for field in EVAL_FIELDS:
        if field in exp:
            expected = exp[field]
            actual = getattr(got, field, None)
            row[field] = "✓" if actual == expected else f"✗ (got {actual!r}, expected {expected!r})"
    results.append(row)

# Print a summary table
import pandas as pd
df = pd.DataFrame(results).set_index("id")
df

,category,alert,temperature_f,breathing_issues,urination_8h,drinking,current_medication,fever,vomiting,rash
id,,,,,,,,,,
tc_01,implied-field,✓,✓,✓,✓,✓,✓,✓,✓,✓
tc_02,ambiguous-value,✓,✓,✓,✓,✓,✓,✓,✓,✓
tc_03,non-overwrite-trap,"✗ (got 'yes', expected None)","✗ (got 101.5, expected None)",✓,✓,"✗ (got 'some', expected None)",✓,"✗ (got 'yes', expected None)","✗ (got 'once', expected None)",✓
tc_04,medication-disambiguation,✓,✓,✓,✓,✓,"✗ (got 'none', expected None)",✓,✓,✓
tc_05,hedge-language,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓,✓,✓,✓
tc_06,multi-field-dump,✓,✓,✓,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓
tc_07,valid-overwrite,✓,✓,✓,✓,✓,✓,✓,"✗ (got 'repeated', expected None)",✓
tc_08,implied-field,✓,✓,✓,✓,✓,✓,✓,✓,✓
tc_09,vague-numeric,✓,"✗ (got 100.0, expected None)",✓,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓


In [15]:
# Aggregate accuracy per field
total = len(results)
per_field = {}
for field in EVAL_FIELDS:
    correct = 0
    asserted = 0
    for row in results:
        if field in row:
            asserted += 1
            if row[field] == "✓":
                correct += 1
    if asserted > 0:
        per_field[field] = (correct, asserted, correct / asserted)

print(f"{'field':22s}  {'correct/total':>14s}  {'accuracy':>10s}")
print("-" * 52)
for field, (c, t, acc) in sorted(per_field.items(), key=lambda x: -x[1][2]):
    print(f"{field:22s}  {c:>6d}/{t:<6d}  {acc*100:>8.1f}%")

# Overall
total_correct = sum(c for c, _, _ in per_field.values())
total_asserted = sum(t for _, t, _ in per_field.values())
print("-" * 52)
print(f"{'OVERALL':22s}  {total_correct:>6d}/{total_asserted:<6d}  {total_correct/total_asserted*100:>8.1f}%")

field                    correct/total    accuracy
----------------------------------------------------
breathing_issues            10/10         100.0%
rash                        10/10         100.0%
alert                        9/10          90.0%
urination_8h                 9/10          90.0%
drinking                     9/10          90.0%
current_medication           9/10          90.0%
temperature_f                8/10          80.0%
vomiting                     8/10          80.0%
fever                        7/10          70.0%
----------------------------------------------------
OVERALL                     79/90          87.8%


## 11. Limitations and future work

**Current limitations**

- **KG substring matching.** David's Neo4j synonym index uses substring containment, so `"fever"` matches both `Fever` and `Febrile convulsion`. The safety layer now gates every KG-sourced red flag through per-`rule_id` predicates against the actual facts, but the long-term fix is to switch to token/phrase matching at the KG layer.
- **Alex's conservative dehydration scoring.** `poor_intake + fever_present` alone is enough to trigger `dehydration_concern → urgent_eval`, even with normal urine output and no vomiting. This is clinically reasonable but strict; many "borderline" home cases land in `urgent_eval`.
- **Interpretation agent never overwrites** — missing fields stay missing across turns. This means a caregiver can't say "actually, he's drinking normally now" and have the system update a previous `intake=reduced`. A future turn-awareness layer should reconcile explicit corrections.
- **Explanation hallucination risk.** The LLM verbalizer is given a strict JSON decision object, but it still generates free text. We rely on the system prompt ("only reword the structured decision") to prevent it from adding clinical judgments. A template-based fallback would be safer.
- **Dosing.** The KG has acetaminophen/ibuprofen dosing concepts but the pipeline doesn't yet surface weight-based dose recommendations. The `weight_kg` field is wired but unused.

**Future work**

1. **Rule + red-flag unification** into a single declarative format (YAML or JSON) so Alex's rules, David's KG rules, and the fallback rules can all be validated against the same test matrix.
2. **Turn-aware interpretation** that distinguishes "new information" from "correction" so the agent can properly update prior facts.
3. **Counterfactual explanations.** "What would need to change for this to be a `home_monitor` case?" — directly leverages the symbolic trace.
4. **End-to-end evaluation.** The current golden dataset covers the interpretation layer only. We need disposition-level golden cases that exercise the whole pipeline.
5. **Safe deployment.** This notebook is a research prototype, not a medical device. For any real deployment we'd need human-in-the-loop oversight, clinician validation of every rule, and clear scoping to "decision support" not "decision-making".

---

**Repo:** `neurosym-finalproject/` · **Package:** `caretrace/` · **Run the interactive CLI:** `python -m caretrace.app`
